# Session 1 · Adapting & Serving LLMs in Production
**PHAROS-CY · WP7 Training · Cyprus University of Technology** · Instructors: Yiannos Vyrides · Kyriakos-Evmenios N. Nomikos

You will fine-tune a small LLM with **QLoRA**, reason about **GRPO** reward design, budget the **KV-cache**, see how
**vLLM** serves adapters, and ground answers with a small **RAG** loop. Every code cell starts with a label (S1, A2, R1 …)
so you can follow the instructor.

| Time | Part | Cells |
|------|------|-------|
| 0:00 | Setup | S1–S2 |
| 0:15 | A · Supervised fine-tuning, LoRA & QLoRA | A1–A6 |
| 1:00 | Break | |
| 1:05 | GRPO & reward design | G1–G3 |
| 1:22 | B · Decoding, KV-cache, vLLM | B1–B2 |
| 1:39 | RAG | R1–R3 |
| 1:51 | Wrap-up · save your adapter for the take-home | A7 |

**How to run.** Colab: *File → Save a copy in Drive*, then *Runtime → Change runtime type → T4 GPU*, then run S1.
The model is `Qwen/Qwen2.5-0.5B-Instruct`, small enough for a free T4; numbers quoted for a 13B model are illustrations.

> **One runtime, one job.** This notebook never installs vLLM — vLLM brings its own torch/CUDA and would break Colab.
> Serving lives in a separate environment: `scripts/serving/serve_vllm_rag.py`.

## Refresher · four ideas
1. **Next token.** logits → softmax → p(x_t | x_<t). Generation = pick a token, append, repeat.
2. **Attention.** softmax(QKᵀ / √d_k)·V in H heads per layer. The K and V of every past token are what the **KV-cache** stores.
3. **Training.** Cross-entropy, forward → backward → optimiser step. Fine-tuning = the same loop on your data.
4. **Train vs infer.** Training sees the whole sequence at once; inference goes token by token — so decoding and the cache matter only when serving.

## Setup

In [ ]:
# S1 · Install pinned versions (keeps Colab's torch/CUDA; never installs vLLM)
# Remove an incompatible optional backend before importing PEFT.
# This notebook uses bitsandbytes NF4, not torchao quantization.
import importlib, importlib.metadata as _md, sys, subprocess
from packaging.version import Version

_removed_torchao = False
try:
    _torchao_version = _md.version("torchao")
except _md.PackageNotFoundError:
    pass
else:
    if Version(_torchao_version) < Version("0.16.0"):
        print("Removing incompatible optional torchao:", _torchao_version)
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
        _removed_torchao = True

%pip install -q \
    "transformers==5.16.1" \
    "trl==1.12.0" \
    "peft==0.20.0" \
    "datasets==5.0.1" \
    "accelerate==1.14.0" \
    "sentence-transformers==6.0.1" \
    bitsandbytes

# Restart once if pyarrow/numpy were upgraded under the running kernel (fixes the pyarrow "size changed" error)
import importlib, importlib.metadata as _md, sys
importlib.invalidate_caches()

_stale = ["torchao (removed)"] if _removed_torchao else []
for _name in ("pyarrow", "numpy"):
    _loaded = sys.modules.get(_name)
    if _loaded is None:
        continue
    try:
        if _md.version(_name) != getattr(_loaded, "__version__", None):
            _stale.append(_name)
    except _md.PackageNotFoundError:
        pass

if _stale:
    print("Runtime package changes:", ", ".join(_stale))
    print("Restarting once so package checks and binary imports reload cleanly.")
    print("→ When it comes back, run this cell again and continue straight down the notebook.")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(True)
else:
    print("Environment ready — no restart needed.")

In [ ]:
# S2 · Imports, seeds, GPU check — use_bf16 drives every dtype below (T4 = sm_75 → fp16)
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    major, minor = torch.cuda.get_device_capability()
    use_bf16 = major >= 8
    gpu_name = torch.cuda.get_device_name(0)
else:
    major, minor, use_bf16, gpu_name = 0, 0, False, None

compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
model_dtype = compute_dtype if device == "cuda" else torch.float32   # CPU fallback: fp32

import pyarrow, transformers, trl, peft, datasets, accelerate
print("torch       ", torch.__version__, "| CUDA:", torch.version.cuda, "| device:", device)
if device == "cuda":
    print("gpu         ", gpu_name, f"(sm_{major}{minor})")
    print("dtype plan  ", f"use_bf16={use_bf16} | compute_dtype={compute_dtype} | "
                          f"model_dtype={model_dtype} | bf16={use_bf16} fp16={not use_bf16}")
print("transformers", transformers.__version__)
print("trl         ", trl.__version__)
print("peft        ", peft.__version__)
print("datasets    ", datasets.__version__)
print("accelerate  ", accelerate.__version__)
print("pyarrow     ", pyarrow.__version__)

if device == "cpu":
    print("\nNo GPU detected. Use Runtime -> Change runtime type -> GPU (a T4 is enough).")
    print("Every cell still runs on CPU, but the 4-bit path is skipped and training is slow.")

## Part A · Supervised fine-tuning (SFT)
SFT is **imitation**: the pretraining loss, on curated (prompt x, response y) pairs, over the response tokens only:

  𝓛_SFT(θ) = − Σ_t log p_θ( y_t | x, y_<t )

- **Loss masking** — prompt tokens get label −100 and add nothing to the loss (`completion_only_loss=True`).
- **Same chat template** for training and inference, or a good model looks broken.
- SFT shapes **behaviour**; facts come from retrieval (RAG, later).

| Knob | Sensible range | Here |
|------|----------------|------|
| Learning rate | 1–2 × 10⁻⁵ | 2e-5 |
| Epochs | 1–3 | 1 |
| Warm-up | a few % of steps | 10 steps |
| Data | diverse formats & tasks | 800 Dolly instructions |

**Two failure modes, one root cause (too little variety, seen too many times):**
*catastrophic forgetting* (fix: 1–2 epochs, lower LR, mix 10–20 % general data) and
*style overfitting* (fix: fewer epochs, diverse formats).

In [ ]:
# A1 · Load 800 Dolly rows as chat prompt/completion pairs
from datasets import load_dataset

N_TRAIN = 800
ds = load_dataset("databricks/databricks-dolly-15k", split="train").select(range(N_TRAIN))

def to_prompt_completion(ex):
    """Dolly row -> TRL conversational prompt/completion (loss on the completion only)."""
    instr = ex["instruction"]
    ctx = ex.get("context", "")
    prompt = instr if not ctx else f"{instr}\n\nContext:\n{ctx}"
    return {
        "prompt":     [{"role": "user",      "content": prompt}],
        "completion": [{"role": "assistant", "content": ex["response"]}],
    }

ds = ds.map(to_prompt_completion, remove_columns=ds.column_names)
print(ds)
print("-" * 60)
print("PROMPT     :", ds[0]["prompt"][0]["content"][:300])
print("COMPLETION :", ds[0]["completion"][0]["content"][:300])

## LoRA & QLoRA
Freeze W and learn a low-rank correction:  h = W·x + (α / r)·B·A·x,  B ∈ ℝ^(d_out×r), A ∈ ℝ^(r×d_in), r ≪ d.

- Only A and B train; **B = 0 at init**, so training starts exactly at the base model.
- Parameters per matrix: r·(d_in + d_out) instead of d_in·d_out — Qwen's `q_proj` (896×896, r = 16): 28,672 vs 802,816.
- Adapted here: q, k, v, o, gate, up, down in all 24 layers · r = 16, α = 32.
- **Wins twice:** a regulariser on small data, and a serving superpower (one base + tiny hot-swappable adapters).

**QLoRA** = the same adapters on a base stored in **4-bit NF4** (≈ 4× smaller); adapters and optimiser stay in higher precision.

In [ ]:
# A2 · NF4 config → model → LoRA → SFTTrainer (+ the T4 bf16 fix at the end)
from transformers import BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

BASE = "Qwen/Qwen2.5-0.5B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb if device == "cuda" else None,   # 4-bit needs CUDA
    dtype=model_dtype,                                       # explicit: "auto" would load bf16 on a T4
    device_map="auto" if device == "cuda" else None,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

trainer = SFTTrainer(
    model=model,
    peft_config=lora,
    train_dataset=ds,
    args=SFTConfig(
        output_dir="./adapter",
        num_train_epochs=1,
        learning_rate=2e-5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # effective batch 16 → 50 steps
        lr_scheduler_type="cosine",
        warmup_steps=10,
        bf16=use_bf16,
        fp16=(device == "cuda" and not use_bf16),
        logging_steps=10,
        completion_only_loss=True,       # loss masking
        max_length=1024,
        report_to="none",
    ),
)

# TRL casts trainable params of a 4-bit model to bf16; a T4 (fp16 GradScaler) cannot unscale bf16 → put them back in fp32
if not use_bf16:
    recast = sum(1 for p in trainer.model.parameters()
                 if p.requires_grad and p.dtype == torch.bfloat16)
    for p in trainer.model.parameters():
        if p.requires_grad and p.dtype == torch.bfloat16:
            p.data = p.data.to(torch.float32)
    print(f"fp16 path: recast {recast} trainable tensors bfloat16 -> float32")

trainable_dtypes = sorted({str(p.dtype) for p in trainer.model.parameters() if p.requires_grad})
print("trainable parameter dtypes:", trainable_dtypes)
assert use_bf16 or "torch.bfloat16" not in trainable_dtypes, (
    "bfloat16 trainable parameters on a GPU whose GradScaler cannot unscale them"
)

In [ ]:
# A3 · One prepared example: chat template applied, prompt tokens masked with -100
ex = trainer.train_dataset[0]
ids, labels = ex["input_ids"], ex["labels"]
n_masked = sum(1 for l in labels if l == -100)
print(tok.decode(ids))
print("-" * 60)
print(f"{len(ids)} tokens in the sequence; {n_masked} prompt/template tokens are masked (label -100),")
print(f"the loss is computed on the remaining {len(ids) - n_masked} -- the assistant turn:")
print(repr(tok.decode([t for t, l in zip(ids, labels) if l != -100])))

In [ ]:
# A4 · Train (≈ 5–10 min on a T4) and save the adapter
trainer.train()

trainer.model.save_pretrained("./adapter")   # a few MB, not a new model
import os
print("Saved LoRA adapter to ./adapter:", sorted(os.listdir("./adapter")))

In [ ]:
# A5 · API drift check — the exact TRL surface this notebook relies on
import inspect, os
import trl
from trl import SFTConfig

params = inspect.signature(SFTConfig).parameters
print("TRL version:                     ", trl.__version__)
print("trainer.model type:              ", type(trainer.model).__name__)
print("SFTConfig has max_length:        ", "max_length" in params, " (old name: max_seq_length)")
print("SFTConfig has warmup_steps:      ", "warmup_steps" in params, " (removed: warmup_ratio)")
print("SFTConfig has completion_only_loss:", "completion_only_loss" in params)
print("adapter written:                 ", os.path.exists("./adapter/adapter_model.safetensors"))

### Exercise (pairs, on paper, while A4 trains) — how big is the adapter?
Sum r·(d_in + d_out) over the 7 adapted modules of one layer, × 24 layers. Qwen2.5-0.5B: hidden 896; 14 query heads and
2 key/value heads of width 64; MLP width 4864; r = 16. What % of 494 M parameters trains?
*Fast pairs:* repeat for a 13B model (d = 5120, 40 heads × 128, no GQA, MLP 13824, 40 layers). Then run A6.

In [ ]:
# A6 · Solution — formula vs PEFT's own count, plus the 13B serving illustration
trainer.model.print_trainable_parameters()   # lives on the PEFT model the trainer wraps

def lora_params(d_model, n_heads, n_kv_heads, head_dim, d_mlp, n_layers, r):
    """r*(d_in + d_out) over the 7 target modules of one layer, × n_layers."""
    d_q, d_kv = n_heads * head_dim, n_kv_heads * head_dim
    per_layer = r * ((d_model + d_q)          # q_proj
                     + 2 * (d_model + d_kv)   # k_proj, v_proj (small with GQA)
                     + (d_q + d_model)        # o_proj
                     + 2 * (d_model + d_mlp)  # gate_proj, up_proj
                     + (d_mlp + d_model))     # down_proj
    return per_layer * n_layers

cfg = trainer.model.config
r = lora.r
n_formula = lora_params(cfg.hidden_size, cfg.num_attention_heads, cfg.num_key_value_heads,
                        cfg.hidden_size // cfg.num_attention_heads, cfg.intermediate_size,
                        cfg.num_hidden_layers, r)
n_trainable, n_all = trainer.model.get_nb_trainable_parameters()
print(f"{BASE}: r={r}  formula={n_formula:,}  trainable={n_trainable:,}  "
      f"-> {100*n_trainable/n_all:.2f}% of all parameters")

# Illustration: a 13B-class model (not the model loaded above)
n_13b = lora_params(5120, 40, 40, 128, 13824, 40, r)
print(f"13B (illustration): LoRA params={n_13b:,}  ~{n_13b*2/1e9:.2f} GB in FP16  "
      f"-> {100*n_13b/13e9:.2f}% of 13e9")

for label, d in [("13B q_proj (illustration)", 5120), ("Qwen2.5-0.5B q_proj (loaded)", 896)]:
    print(f"{label}: dense d*d = {d*d:>10,}   LoRA 2*d*r = {2*d*r:>8,}"
          f"  ->  {100*2*d*r/(d*d):.2f}% of the dense update")

base_gb, adapter_gb = 26, 0.1   # 13B in FP16 ≈ 26 GB; one adapter ≈ 0.1 GB
print(f"\n3 full 13B copies: {3*base_gb} GB   vs   1 base + 3 adapters: {base_gb + 3*adapter_gb:.1f} GB")

## From imitation to optimisation — GRPO
SFT can only imitate. Correctness and safety are not clean (input, output) pairs — but we can **score** them.
For each prompt sample a group of G answers, reward each, and standardise **within the group**:

  A_ij = ( r_ij − μ_i ) / ( σ_i + ε )

The group is the baseline, so no critic network. The update is a PPO-style clipped ratio plus β·KL to the SFT model.
We do not run the full loop today; G1–G3 isolate the ideas that decide whether it works.

**Why divide by σ?** So every prompt gets a vote: a high-variance prompt cannot drown out a low-variance one. *Predict first:*
which of the two prompts in G1 dominates if we only subtract the mean?

In [ ]:
# G1 · Group-relative advantage, with and without ÷σ
import numpy as np

def group_advantage(rewards, eps=1e-6, divide_by_sigma=True):
    """A_ij = (r_ij - mu_i) / (sigma_i + eps) for one prompt's group of rewards."""
    rewards = np.asarray(rewards, float)
    centred = rewards - rewards.mean()
    return centred / (rewards.std() + eps) if divide_by_sigma else centred

groups = {
    "safety-critical prompt": [0.5, 5.0, 1.0, 4.5, 2.0, 5.0],   # excellent → dangerous
    "simple definition     ": [4.0, 4.2, 4.5, 4.1, 4.3, 4.0],   # all roughly fine
}
for name, r in groups.items():
    r = np.asarray(r, float)
    print(f"{name}: mean={r.mean():.2f}  std={r.std():.2f}")
    print("   mean-subtracted only:", np.round(group_advantage(r, divide_by_sigma=False), 2))
    print("   standardised (GRPO) :", np.round(group_advantage(r), 2))

safety, define = groups.values()
raw = [np.abs(group_advantage(g, divide_by_sigma=False)).max() for g in (safety, define)]
std = [np.abs(group_advantage(g)).max() for g in (safety, define)]
print(f"\nlargest |advantage|, safety vs definition -- without /sigma: {raw[0]:.2f} vs {raw[1]:.2f}"
      f" ({raw[0]/raw[1]:.0f}x)  |  with /sigma: {std[0]:.2f} vs {std[1]:.2f}")

### Reward design & reward hacking
The policy optimises the **metric**, not the intent. Blend r = 0.7·correct + 0.3·safe and a correct-but-dangerous
answer (5, 1) scores 3.8 — it looks fine and ships. Keep signals **separate** and put a **floor** on the non-negotiable
one (safety < 4 ⇒ −5). KL alone will not stop hacking.

In [ ]:
# G2 · Separate signals + a safety floor, then standardise per group
def reward(correctness, safety, alpha=0.7, safety_floor=4.0):
    r = alpha*correctness + (1-alpha)*safety
    return np.where(safety < safety_floor, r - 5.0, r)     # hard penalty below the floor

drug   = reward(np.array([5,4,1,2]), np.array([5,2,5,1]))   # answer 2: fairly correct but unsafe
define = reward(np.array([5,5,4,5]), np.array([5,5,5,5]))

print("drug   rewards:", np.round(drug, 2), " spread:", round(float(drug.std()), 2))
print("define rewards:", np.round(define, 2), " spread:", round(float(define.std()), 2))
print()
print("drug   group advantages:", np.round(group_advantage(drug), 2))
print("define group advantages:", np.round(group_advantage(define), 2))

### Exercise G3 · Hack a grounding reward, then fix it *(research extension · ~10 min · CPU only)*
Suppose we want GRPO to make the RAG model of Part B more honest. A first attempt rewards **citing** (+2),
**brevity** (+1) and punishes **giving up** (−3). G3 scores four sampled answers per prompt with that reward and turns
them into GRPO advantages with `group_advantage` from G1.

1. **Predict** which answer the policy is pushed towards for the unanswerable prompt — then run G3.
2. Write `fixed_grounding_reward(answer, n_docs)` in G3: keep the signals separate, give a citation credit only when the
   cited passage really exists (`index < n_docs`), and put a **floor** (−5) on fabrication: citing with no evidence, or
   answering when `n_docs == 0`.
3. Re-run G3 and check that the honest answer now gets the largest advantage in **both** groups.
4. **Hack your own fix:** find an answer string that scores highest under it without saying anything useful.
   What does that tell you about rewarding *citations* rather than *support*? (Compare with guardrail 3 in Part B.)

Then run **G3 ✓** for one possible solution. The take-home *stretch* builds on this cell.

In [ ]:
# G3 · Exercise — a naive grounding reward, scored as GRPO advantages
import re

def naive_grounding_reward(answer, n_docs):
    """First attempt: reward citing and brevity, punish giving up. n_docs = passages that passed the gate."""
    a = answer.strip()
    r = 0.0
    r += 2.0 if re.search(r"\[\d+\]", a) else 0.0      # cites something
    r += 1.0 if len(a.split()) <= 25 else 0.0          # brief
    r -= 3.0 if a.lower().startswith("unknown") else 0.0
    return r

def fixed_grounding_reward(answer, n_docs):
    """TODO (step 2): separate signals, valid citations only, floor of -5 on fabrication."""
    return naive_grounding_reward(answer, n_docs)       # replace this line

# Four sampled answers per prompt (n_docs = passages that passed the retrieval gate)
G3_GROUPS = {
    "Who painted the Mona Lisa?  (n_docs=0)": (0, [
        "unknown",
        "Leonardo da Vinci painted the Mona Lisa. [0]",
        "It was painted in the 16th century by an Italian artist.",
        "[0]",
    ]),
    "What is the capital of Cyprus?  (n_docs=1)": (1, [
        "Nicosia is the capital of Cyprus. [0]",
        "unknown",
        "Nicosia. [3]",
        "The capital is Nicosia.",
    ]),
}

def score_groups(reward_fn, title):
    print(f"── {title}")
    for prompt, (n_docs, answers) in G3_GROUPS.items():
        r = np.array([reward_fn(a, n_docs) for a in answers])
        adv = group_advantage(r)
        best = int(np.argmax(adv))
        print(f"  {prompt}")
        for a, ri, ai in zip(answers, r, adv):
            print(f"     reward={ri:+5.1f}  advantage={ai:+5.2f}  {'◀ pushed towards' if a == answers[best] else '':16s} {a!r}")
    print()

score_groups(naive_grounding_reward, "naive reward")
score_groups(fixed_grounding_reward, "your fixed reward")

In [ ]:
# G3 · One possible solution — run after you have tried
def fixed_grounding_reward(answer, n_docs):
    a = answer.strip()
    cites = [int(i) for i in re.findall(r"\[(\d+)\]", a)]
    abstained = a.lower().startswith("unknown")
    valid = bool(cites) and all(i < n_docs for i in cites)

    if n_docs == 0:                       # no evidence: only "unknown" is acceptable
        return 3.0 if abstained else -5.0
    if cites and not valid:               # cites a passage that is not there = fabrication
        return -5.0
    r_cite    = 2.0 if valid else 0.0     # signals kept separate ...
    r_abstain = -1.0 if abstained else 0.0
    r_brief   = 0.5 if len(a.split()) <= 40 else 0.0
    return r_cite + r_abstain + r_brief   # ... and readable one by one

score_groups(fixed_grounding_reward, "solution reward")
print("Hack of the fix:", repr("[0]"), "→", fixed_grounding_reward("[0]", 1),
      "— a bare citation scores like a real answer. Rewarding citations ≠ rewarding support.")

## Part B · Serving at scale
**Decoding** — how the next token is picked from p = softmax(ℓ):

| Rule | What it does | Use it for |
|------|--------------|------------|
| Greedy | argmax every step | factual QA, our RAG cells |
| Beam search | keep the k best sequences | translation, targeted tasks |
| Temperature T | ℓ / T: < 1 sharpens, > 1 flattens | a knob before sampling |
| Top-k | sample from the k most likely tokens | open-ended chat |
| Top-p (nucleus) | smallest set with mass ≥ p | open-ended chat, code |

For code: top-p sampling + best-of-N with tests beats beam — one correct candidate is enough.

**KV-cache** — one K and one V per token, per layer, per KV head:  M ≈ 2 · S · L · H_kv · d_h · b/8 bytes per sequence.
A 13B model in FP16 at 4,096 tokens needs ≈ 3.36 GB *per sequence* next to 26 GB of weights; 64 users ≈ 215 GB.
Shrink it with **INT8** (½) or **GQA** (fewer KV heads — Qwen2.5-0.5B has 14 query heads but 2 KV heads).

In [ ]:
# B1 · KV-cache budget: 13B illustration, then the model you fine-tuned (GQA)
def kv_cache_gb(S, L, H_kv, d_h, bits):
    """KV-cache size for ONE sequence, in GB. H_kv = number of key/value heads."""
    return (2 * S * L * H_kv * d_h * (bits/8)) / 1e9

print("13B FP16, S=4096:", round(kv_cache_gb(4096, 40, 40, 128, 16), 2), "GB / sequence")
print("13B INT8, S=4096:", round(kv_cache_gb(4096, 40, 40, 128, 8), 2), "GB / sequence")
print("FP16 x 64 seqs  :", round(64*kv_cache_gb(4096, 40, 40, 128, 16), 1), "GB")
print()

cfg = trainer.model.config
L      = cfg.num_hidden_layers
H_q    = cfg.num_attention_heads
H_kv   = getattr(cfg, "num_key_value_heads", H_q)
d_h    = getattr(cfg, "head_dim", None) or cfg.hidden_size // H_q
print(f"{BASE}: layers={L}, query heads={H_q}, KV heads={H_kv}, head_dim={d_h}")
print("  with GQA        :", round(kv_cache_gb(4096, L, H_kv, d_h, 16) * 1e3, 1), "MB / sequence")
print("  if it had no GQA:", round(kv_cache_gb(4096, L, H_q,  d_h, 16) * 1e3, 1), "MB / sequence")
print(f"  -> GQA saves {H_q // H_kv}x on the cache, for free.")

### vLLM & PagedAttention
A naive server reserves one contiguous worst-case block per request — mostly empty. **PagedAttention** hands out small
blocks on demand (< 4 % waste); **continuous batching** lets requests join as others finish. The vLLM paper (Kwon et al.,
2023) reports up to ≈ 24× a naive `generate` loop, and `enable_lora` serves a different adapter per request.
For inference, data parallelism replicates a model that fits on each GPU; tensor and pipeline parallelism split a larger model across GPUs. FSDP shards training state and is a separate training choice.

> **Never `pip install vllm` in this Colab** — it replaces PyTorch. B2 only *prints* the serving code;
> run it with `scripts/serving/serve_vllm_rag.py` in a separate environment, carrying `./adapter` across.

In [ ]:
# B2 · vLLM serving reference — printed, not executed (separate environment)
vllm_example = r"""
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="float16",                 # "bfloat16" on Ampere or newer
    gpu_memory_utilization=0.80,
    enable_prefix_caching=True,      # reuse the KV-cache across shared prefixes
    max_lora_rank=16,                # matches the training adapter
    enable_lora=True,                # hot-swap adapters per request
    max_model_len=2048,
    # tensor_parallel_size=2,        # uncomment for 2-GPU tensor parallelism
)

sp = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=128)
tokenizer = llm.get_tokenizer()
prompt_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Explain LoRA in two sentences."}],
    tokenize=True, add_generation_prompt=True,
)
out = llm.generate(
    [{"prompt_token_ids": prompt_ids}],
    sp,
    lora_request=LoRARequest("session1-adapter", 1, "./adapter"),
)
print(out[0].outputs[0].text)
"""

print(vllm_example)
print("Reference only — see scripts/serving/serve_vllm_rag.py. This runtime keeps Colab's training stack intact.")

## RAG · grounding answers in facts
Fine-tuning teaches a *skill*, not *facts*. Retrieve at query time and answer **only** from the context — or say *unknown*.
Pipeline: embed (MiniLM, 384-d) → cosine similarity → top-k → cited prompt → generate → check.

**Three guardrails — retrieval alone does not ground a model:**
1. **Gate** (`MIN_SIM = 0.35`): top-k always returns *something*; the Mona Lisa's best match scores ≈ 0.1.
2. **Copy-friendly decoding**: greedy, `repetition_penalty=1.0` (Qwen ships 1.1, which fights copying).
3. **Citation check**: an answer is shown only if it cites a passage that is really in the context.

Start **R1** now — it downloads the embedding model while you read.

In [ ]:
# R1 · RAG pipeline: retrieve → gate → cited prompt → generate → citation check (+ Case B)
import re
from sentence_transformers import SentenceTransformer, util

corpus = [
    "LoRA freezes W and learns a low-rank BA update; only 2*d*r parameters train.",
    "PagedAttention stores the KV-cache in non-contiguous blocks to cut waste.",
    "GRPO standardises rewards within a group so different reward scales contribute comparably.",
    "Dividing by the group standard deviation puts every prompt group on the same scale, so a "
    "high-variance group cannot dominate the update and a low-variance group still teaches the model.",
    "Nucleus sampling draws from the smallest set of tokens whose probability mass exceeds p.",
    "Nicosia is the capital of Cyprus.",
]

embed = SentenceTransformer("all-MiniLM-L6-v2")
C = embed.encode(corpus, convert_to_tensor=True)

MIN_SIM = 0.35   # answer-bearing passages score ~0.45–0.85 here, unrelated ones ~0.0–0.2

def retrieve(query, k=2, min_sim=None):
    """Top-k passages that pass the gate -> [(text, score)] in rank order."""
    if k < 1:
        raise ValueError("k must be positive")
    if min_sim is None:
        min_sim = MIN_SIM
    q = embed.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q, C)[0]
    top = scores.topk(min(k, len(corpus)))
    return [(corpus[int(i)], float(s)) for s, i in zip(top.values, top.indices) if float(s) >= min_sim]

def build_prompt(query, docs):
    """Use numbered passages, preserve their meaning, and abstain without evidence."""
    context = "\n".join(f"[{i}] {d}" for i, (d, _) in enumerate(docs)) or "(no relevant passages found)"
    return (
        "Answer using only the numbered context passages below. Preserve their meaning.\n"
        "Give one or two short sentences and finish with the citation of the passage used, such as [0].\n"
        "If the context does not answer the question, reply with exactly: unknown\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    )

def answer(query, k=2, max_new_tokens=96):
    """Returns (prompt, displayed_answer, raw_output); displayed = raw only if it cites a real passage."""
    docs = retrieve(query, k)
    prompt = build_prompt(query, docs)
    chat = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)   # same template as training
    inputs = tok(text, return_tensors="pt", add_special_tokens=False).to(trainer.model.device)

    trainer.model.eval()
    trainer.model.config.use_cache = True          # the Trainer turned it off
    with torch.no_grad():
        generated = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                       # greedy
            repetition_penalty=1.0,                # copy-friendly
            pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
        )
    raw = tok.decode(generated[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    cited = [int(i) for i in re.findall(r"\[(\d+)\]", raw)]
    # This checks citation IDs only; it does not verify that the text supports the answer.
    grounded = bool(cited) and all(i < len(docs) for i in cited) and not raw.lower().startswith("unknown")
    return prompt, (raw if grounded else "unknown"), raw

def show(query, k=2):
    """Trace one question: retrieved passages and gate decision, raw output, displayed answer."""
    prompt, final, raw = answer(query, k)
    print("Q:", query)
    for i, (d, s) in enumerate(retrieve(query, k, min_sim=-1.0)):
        print(f"   top-{i} cos_sim={s:.2f} {'-> context [' + str(i) + ']' if s >= MIN_SIM else '-> below gate, dropped'}: {d}")
    print("   raw model output:", repr(raw))
    print("   displayed answer:", final)

# Case B — answered by the course-material passages
query = "Why does GRPO divide by the group standard deviation?"
prompt, grounded, raw = answer(query)
print("=== Prompt sent to the model ===")
print(prompt)
print("\n=== Displayed answer ===")
print(grounded)

In [ ]:
# R2 · Case A — one passage answers it: the fact must come from the context, with its citation
show("What is the capital of Cyprus?")

In [ ]:
# R3 · Case C — nothing passes the gate: the answer must be "unknown" (the citation check catches a bluff)
show("Who painted the Mona Lisa?")

### Exercise · pick one (2 min)
1. `show(query, k=3)` — does a third passage help or dilute?
2. Remove `Preserve their meaning.` from `build_prompt`, re-run R1–R3. Does the answer remain faithful to the passages?
3. Move the answer instructions **below** the context and question, then re-run R1–R3. Does the model still cite?
4. Set `MIN_SIM = 0` and re-run R3 — irrelevant passages now enter the prompt. Does the model cite them?
5. `show("What is the capital of Greece?")` — the Cyprus passage passes the gate (≈ 0.5) but does not answer. The hard case.
6. Add one fact of your own to `corpus` (re-run R1) and ask a question only it can answer.

## Wrap-up
- Fewer epochs + diverse data beat brute-force SFT; **LoRA is a regulariser and a serving superpower.**
- GRPO: standardise per group; keep reward signals separate and floor the non-negotiable.
- Budget the **KV-cache** before you deploy; page it with vLLM; pick decoding by task.
- Grounding = retrieval + gate + copy-friendly decoding + citation check — and let the model say *unknown*.
- Keep the **training** and **vLLM serving** environments separate.

**Next → Session 2 · Multimodal & Diffusion.**

## Take-home — in a separate notebook
The assignment *Grounding under the microscope* is described in **`assignment/Assignment.pdf`**.
Use the separate assignment Colab link supplied by your instructor. Run **A7 before you close this session**:
it downloads `adapter.zip`, which you upload to the assignment runtime before running L1.


In [ ]:
# A7 · Save the adapter for the take-home (downloads adapter.zip; optional Google Drive copy)
import os, shutil, zipfile
from pathlib import Path

adapter_dir = Path("./adapter")
adapter_files = ["adapter_config.json", "adapter_model.safetensors"]
missing = [name for name in adapter_files if not (adapter_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Run A4 to train and save the adapter first. Missing: " + ", ".join(missing)
    )

# Export the adapter only, leaving training checkpoints and optimizer state out.
with zipfile.ZipFile("adapter.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in adapter_files:
        archive.write(adapter_dir / name, arcname=name)
print("adapter.zip:", round(os.path.getsize("adapter.zip") / 1e6, 1), "MB")

SAVE_TO_DRIVE = False   # True -> also copy the adapter to MyDrive/session1_adapter
try:
    from google.colab import files
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_dir = Path("/content/drive/MyDrive/session1_adapter")
        drive_dir.mkdir(parents=True, exist_ok=True)
        for name in adapter_files:
            shutil.copy2(adapter_dir / name, drive_dir / name)
        print("Copied adapter to MyDrive/session1_adapter")
    files.download("adapter.zip")
except ImportError:
    print("Not running on Colab — adapter.zip is in the working directory.")
